# pdf_vlm — Colab runner

Local multimodal document QA stack: **Gemma 3 4B (GGUF)** + OCR/PDF text + page/hierarchical RAG eval.

**Runtime:** Runtime → Change runtime type → **GPU (T4)** recommended for generation.

This notebook:
1. Clones [mAn-He/pdf_vlm](https://github.com/mAn-He/pdf_vlm)
2. Installs the package (light path; hash embedder to avoid BGE-M3 OOM)
3. Ingests Hyundai WIA length packs already in `data/custom/`
4. Runs the eval harness (dry-run by default; full gen after HF login + GGUF download)

## 0. Clone & install

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/mAn-He/pdf_vlm.git"
ROOT = Path("/content/pdf_vlm")

if not (ROOT / "pyproject.toml").exists():
    !git clone --depth 1 {REPO_URL} {ROOT}
else:
    print("Repo already present:", ROOT)

%cd {ROOT}
print("cwd:", Path.cwd())

In [ ]:
# Core + index (faiss) + viz. OCR extras are heavy; Colab path uses stub OCR + PDF text.
!pip -q install -U pip setuptools wheel
!pip -q install -e ".[index,viz,dev]"

# llama-cpp with CUDA wheel when possible (falls back to CPU pip package)
import subprocess, sys

def try_install_llama():
    cmds = [
        [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu124"],
        [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python", "--extra-index-url", "https://abetlen.github.io/llama-cpp-python/whl/cu122"],
        [sys.executable, "-m", "pip", "install", "-q", "llama-cpp-python"],
    ]
    for cmd in cmds:
        print("Trying:", " ".join(cmd[-4:]))
        r = subprocess.run(cmd)
        if r.returncode == 0:
            return True
    return False

ok = try_install_llama()
print("llama-cpp-python installed:", ok)

import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

## 1. (Optional) Hugging Face login + download Gemma GGUF

1. Accept license: https://huggingface.co/google/gemma-3-4b-it-qat-q4_0-gguf
2. Paste a HF token with read access (Colab secrets key `HF_TOKEN` preferred).

In [ ]:
from google.colab import userdata
import os

DOWNLOAD_GGUF = False  # set True after accepting license + adding HF_TOKEN

if DOWNLOAD_GGUF:
    try:
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        from getpass import getpass
        os.environ["HF_TOKEN"] = getpass("HF token: ")
    !huggingface-cli login --token "$HF_TOKEN" --add-to-git-credential
    !python scripts/download_models.py --with-mmproj
else:
    print("Skipping GGUF download. Harness will use dry_run / retrieval-only.")

## 2. Ingest custom packs + build hash indexes

Repo already includes Hyundai WIA truncated PDFs under `data/custom/{5,20,50,100}/`.
Start with **5 + 20** pages on free Colab; raise buckets if RAM allows.

In [ ]:
BUCKETS = "5,20"  # e.g. "5,20,50,100"

!python scripts/colab_prepare_custom.py --buckets {BUCKETS} --stub --enrich-pdf-text --hash-embedder --force

from pdf_vlm.utils.io import load_json, resolve_path
print(load_json(resolve_path("data/custom/colab_prepared.json")))

## 3. Run evaluation harness

`eval_hw_wia_colab.yaml` defaults to `dry_run: true` (retrieval metrics only).
After GGUF download, set `DRY_RUN = False` below.

In [ ]:
DRY_RUN = True  # False only if models/*.gguf exist

cmd = [
    "python", "scripts/run_eval_harness.py",
    "--config", "configs/experiments/eval_hw_wia_colab.yaml",
]
if DRY_RUN:
    cmd.append("--dry-run")

import subprocess
print(" ".join(cmd))
subprocess.check_call(cmd)

In [ ]:
from pathlib import Path
import json

runs = sorted(Path("results/runs").glob("eval_*"), key=lambda p: p.stat().st_mtime, reverse=True)
print("latest run:", runs[0] if runs else None)
if runs:
    summary = runs[0] / "summary.json"
    if summary.exists():
        print(json.dumps(json.loads(summary.read_text(encoding="utf-8")), ensure_ascii=False, indent=2)[:4000])
    else:
        print("files:", list(runs[0].iterdir())[:20])

## 4. (Optional) Inference practicality bench

In [ ]:
from pathlib import Path

gguf = Path("models/gemma-3-4b-it-q4_0.gguf")
if gguf.exists():
    !python scripts/bench_gemma_inference.py --repeats 2
else:
    print("Skip bench: GGUF missing. Enable DOWNLOAD_GGUF in section 1.")

## Notes

| Topic | Colab tip |
|---|---|
| Embeddings | Default **hash** embedder avoids ~2GB BGE-M3 OOM |
| OCR | Stub + PDF text layer; install `.[ocr]` only if you need PP-StructureV3 |
| Multimodal gen | Needs `mmproj-model-f16-4B.gguf` + CUDA llama-cpp |
| Full length | Raise `BUCKETS` to `5,20,50,100` and use `eval_hw_wia.yaml` |

Open in Colab: upload this notebook or use
`https://colab.research.google.com/github/mAn-He/pdf_vlm/blob/main/notebooks/pdf_vlm_colab.ipynb`